# Observatório da Qualidade do Transporte Aéreo

**Execução gratuita no navegador — CPU, sem GPU e sem instalar nada no seu computador.**

Este notebook baixa um mês do VRA/ANAC, valida o CSV, constrói os modelos dbt, roda testes e exporta resultados. A primeira parte usa DuckDB no ambiente remoto do Colab e não pede cartão nem GCP. BigQuery real está em uma etapa opcional separada, que exige autenticação do proprietário.

Comece com janeiro de 2025. O ano completo está disponível alterando a lista de competências. Não há dados fictícios misturados aos dados oficiais: a fixture sintética serve exclusivamente à integração contínua.

**Limite:** o Colab pode encerrar sessões e apagar arquivos locais. Guarde os resultados que quiser conservar. O notebook não cria cobrança nem inicia trial.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

TEST_MODE = os.environ.get("OBS_NOTEBOOK_TEST_MODE") == "1"
REPO = Path(os.environ.get("OBS_REPO_ROOT", "/content/observatorio-qualidade-transporte-aereo")).resolve()
ROOT = Path(os.environ.get("OBS_DATA_ROOT", str(REPO / "data/colab"))).resolve()
os.environ["DBT_SEND_ANONYMOUS_USAGE_STATS"] = "false"
os.environ["DO_NOT_TRACK"] = "1"
if not TEST_MODE:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AlexandreSantosRodrigues/observatorio-qualidade-transporte-aereo.git", str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO) + "[dev]"], check=True)
os.chdir(REPO)
print("Versão do projeto:")
if not TEST_MODE:
    subprocess.run(["git", "rev-parse", "HEAD"], check=True)
print("Preparado. Modo:", "validação automatizada do notebook" if TEST_MODE else "dados oficiais da ANAC")

## 1. Escolha o período

O padrão é um mês para conferir o fluxo. Para o ano completo, substitua a lista pela expressão comentada. Execute sequencialmente e use somente competências disponíveis na fonte. Selecionar o ano pode levar vários minutos na primeira execução.

In [ ]:
COMPETENCIAS = ["2025-01"]
# COMPETENCIAS = [f"2025-{mes:02d}" for mes in range(1, 13)]
if TEST_MODE:
    COMPETENCIAS = ["2025-01"]
for competencia in COMPETENCIAS:
    command = [sys.executable, "-m", "observatorio.cli", "ingest", "--competencia", competencia, "--root", str(ROOT)]
    if TEST_MODE:
        command += ["--arquivo", os.environ["OBS_TEST_SOURCE"]]
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    receipt = json.loads(result.stdout)
    print(competencia, "| fonte:", receipt["source_rows"], "| aceitas:", receipt["accepted_rows"],
          "| quarentena:", receipt["quarantine_rows"], "| duplicatas exatas:", receipt["exact_duplicate_rows"])
    print("Motivos de quarentena:", receipt["quarantine_reasons"])

## 2. Construa e valide os modelos

Os 7 modelos dbt incluem dimensões, fato e agregado mensal. Os testes verificam integridade, relacionamentos, elegibilidade e reconciliação. Uma falha interrompe a execução e impede a publicação de um novo conjunto de exports.

In [ ]:
subprocess.run([sys.executable, "-m", "observatorio.workflow", "--root", str(ROOT)], check=True)
pointer = json.loads((ROOT / "exports/CURRENT.json").read_text())
EXPORTS = ROOT / "exports" / pointer["run_id"]
resumo = json.loads((EXPORTS / "resumo.json").read_text())
print(json.dumps(resumo, ensure_ascii=False, indent=2))

## 3. Explore os resultados

Atraso é definido **neste projeto** como diferença estritamente maior que 15 minutos. Cancelamentos não entram no denominador de pontualidade. Voos realizados sem os horários necessários continuam no fato e são excluídos somente do indicador correspondente. Os horários da fonte são civis, em Brasília.

O destaque GIG/SDU considera Galeão e Santos Dumont; não representa todos os aeroportos do estado do Rio. As classificações de atraso publicadas pela ANAC podem usar outra régua. Não atribuímos causas a atrasos.

In [ ]:
import pandas as pd
from IPython.display import display
rotas = pd.read_csv(EXPORTS / "mart_rotas_mensal.csv")
print("Rotas/companhias com pelo menos 30 chegadas elegíveis:")
display(rotas[rotas.elegiveis_chegada >= 30].sort_values("taxa_atraso_chegada_15", ascending=False).head(15))
print("Galeão e Santos Dumont:")
display(rotas[rotas.envolve_gig_sdu == True].head(15))
print("Arquivos disponíveis:")
for file in sorted(EXPORTS.iterdir()):
    print(file.name, f"{file.stat().st_size / 1024 / 1024:.2f} MiB")

## 4. Confira os testes de engenharia

Incluem virada de dia, identificadores alfanuméricos, reprocessamento, dados corrompidos e cenários com erros deliberados. Um teste negativo passa quando o pipeline detecta e rejeita o erro esperado.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 5. BigQuery real — opcional e separado

Crie um projeto no BigQuery Sandbox **sem vincular faturamento**. Consulte `docs/BIGQUERY_E_IAM.md`. A etapa abaixo não executa por padrão.

A integração consulta a Cloud Billing API e só continua se `billingEnabled=false`. Se essa API não estiver acessível, habilite-a no projeto e conceda somente o acesso de leitura necessário; não retire o bloqueio. O notebook não abre trial, não aceita cartão, não cria chaves de serviço e não transfere credenciais ao GitHub.

A carga usa lotes, tabelas e views; não usa `MERGE`, `INSERT`, snapshots nem streaming. A compatibilidade ponta a ponta com o sandbox precisa ser confirmada pela execução na sua conta. Mesmo sem cobrança, armazenamento e consultas têm limites e expiram. Leia os limites atuais antes de executar o ano completo.

In [ ]:
EXECUTAR_BIGQUERY = False
GCP_PROJECT_ID = ""  # ID do seu projeto sandbox, sem faturamento.
if EXECUTAR_BIGQUERY:
    if not GCP_PROJECT_ID:
        raise ValueError("Preencha o ID do projeto sandbox.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO) + "[bigquery]"], check=True)
    from google.colab import auth
    auth.authenticate_user()
    subprocess.run([sys.executable, "-m", "observatorio.bigquery", "--project", GCP_PROJECT_ID, "--root", str(ROOT)], check=True)
else:
    print("BigQuery não executado. Os resultados anteriores foram calculados com DuckDB.")

## 6. Relatório e próximos usos

Os CSVs de dimensões/fato/agregado podem alimentar Power BI. O guia `docs/POWER_BI.md` explica relacionamentos e medidas DAX. A criação no Power BI web depende da sua conta; esse requisito permanece pendente até um relatório real ser construído.

Para salvar resultados pelo navegador, execute opcionalmente a célula abaixo. Ela cria um ZIP com os exports atuais, sem arquivos de autenticação. O download vai ao seu computador; pode ser grande se incluir o fato do ano inteiro.

Documentação dbt: abra `dbt/target/static_index.html`. Dados rejeitados: `data/.../releases/<competência>/<release>/quarentena.json`. Não publique tokens, credenciais ou arquivos pessoais.

In [ ]:
BAIXAR_RESULTADOS = False
if BAIXAR_RESULTADOS:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(ROOT / "resultados"), "zip", EXPORTS)
    files.download(archive)
else:
    print("Execução concluída. Exports:", EXPORTS)